# **Decoder Casual Language Model**

In [2]:
import torchtext
import torch
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator
from torchtext.datasets import IMDB

### **Dataset**

In [5]:
train_iter,test_iter = IMDB()
next(iter(train_iter))

(1,
 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far betwee

In [10]:
# define device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device.type

'cpu'

## Preprocessing Data

This section focuses on preparing text data for NLP tasks, mainly through tokenization and vocabulary creation.

### Special Tokens and Indices
Defines special tokens such as `<unk>`, `<pad>`, and an empty string (used as an EOS token), and assigns them indices (`0`, `1`, and `2`). These tokens serve specific purposes in text processing:

- `UNK_IDX`: Represents unknown or out-of-vocabulary words  
- `PAD_IDX`: Used to pad shorter sequences in a batch for uniform length  
- `EOS_IDX`: Indicates the end of a sequence  

### `yield_tokens` Function
A generator function that iterates through the dataset (`data_iter`), applies a tokenizer to each sample, and yields tokenized outputs one at a time.

### Building the Vocabulary
Constructs a vocabulary from the tokenized data using `build_vocab_from_iterator`. Special tokens are added at the beginning, and all tokens with a minimum frequency of 1 are included.

### Handling Unknown Tokens
Sets a default index (`UNK_IDX`) for tokens not found in the vocabulary, ensuring the model can handle unseen words gracefully.

### `text_to_index` Function
Converts raw text into a sequence of numerical indices based on the vocabulary, making it suitable as input for machine learning models.

### `index_to_en` Function
Transforms a sequence of indices back into readable text, which is useful for interpreting model outputs.

### Functionality Check
Validates the preprocessing pipeline by converting a sample tensor `[0, 1, 2]` back into the corresponding special tokens, ensuring that the mappings are working correctly.

In [11]:
UNK_IDX, PAD_IDX, EOS_IDX = 0, 1, 2
special_tokens = ["<unk>","<pad>", "<eos>" ]

In [ ]:
tokenizer = get_tokenizer("basic_english")

def yield_tokens(dataset):
    for id, text in dataset:
        yield tokenizer(text)
        
vocab = build_vocab_from_iterator(yield_tokens(train_iter),specials=special_tokens,
                                  special_first=True)
vocab.set_default_index(UNK_IDX)